<a href="https://colab.research.google.com/github/ichameisyak-sudo/analisis-kualitas-udara-indonesia/blob/main/code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Install dan konfigurasi**

In [8]:
import io, re, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests

START, END = "2023-01-01", "2025-12-31"
FORCE = False

DATA = Path("data")
RAW, PROC = DATA / "raw", DATA / "processed"
RAW.mkdir(parents=True, exist_ok=True)
PROC.mkdir(parents=True, exist_ok=True)

CITIES = {
    "Jakarta":  (-6.2088, 106.8456, "Asia/Jakarta",  "DKI Jakarta"),
    "Bandung":  (-6.9175, 107.6191, "Asia/Jakarta",  "Jawa Barat"),
    "Semarang": (-6.9667, 110.4167, "Asia/Jakarta",  "Jawa Tengah"),
    "Surabaya": (-7.2575, 112.7521, "Asia/Jakarta",  "Jawa Timur"),
    "Medan":    (3.5952,   98.6722, "Asia/Jakarta",  "Sumatera Utara"),
    "Makassar": (-5.1477, 119.4327, "Asia/Makassar", "Sulawesi Selatan"),
}

AQ_URL = "https://air-quality-api.open-meteo.com/v1/air-quality"
WX_URL = "https://archive-api.open-meteo.com/v1/archive"
AQ_VARS = ["pm2_5", "pm10", "carbon_monoxide", "nitrogen_dioxide",
           "sulphur_dioxide", "ozone", "aerosol_optical_depth", "us_aqi"]
WX_VARS = ["temperature_2m", "relative_humidity_2m", "precipitation",
           "wind_speed_10m", "wind_direction_10m", "surface_pressure",
           "cloud_cover"]
BPS_COLS = ["provinsi", "tahun", "mobil_penumpang", "bus", "truk", "sepeda_motor"]
YEARS = list(range(int(START[:4]), int(END[:4]) + 1))

**Fungsi unduh Open-Meteo**

In [9]:
def get_json(url, params, retries=5):
    for k in range(retries):
        r = requests.get(url, params=params, timeout=60)
        if r.status_code == 200:
            return r.json()
        if r.status_code in (429, 500, 502, 503, 504):
            time.sleep(2 ** k * 5)
            continue
        raise RuntimeError(f"HTTP {r.status_code}: {r.text[:200]}")
    raise RuntimeError("Gagal setelah beberapa percobaan")


def fetch_city(url, variables, lat, lon, tz, extra=None):
    frames = []
    for year in YEARS:
        params = dict(latitude=lat, longitude=lon,
                      start_date=max(START, f"{year}-01-01"),
                      end_date=min(END, f"{year}-12-31"),
                      hourly=",".join(variables), timezone=tz)
        params.update(extra or {})
        frames.append(pd.DataFrame(get_json(url, params)["hourly"]))
        time.sleep(1)
    df = pd.concat(frames, ignore_index=True)
    df["time"] = pd.to_datetime(df["time"])
    return df

**Jalankan Pengambilan Data**

In [10]:
for city, (lat, lon, tz, _) in CITIES.items():
    for tag, url, vars_, extra in (
        ("aq", AQ_URL, AQ_VARS, None),
        ("wx", WX_URL, WX_VARS, {"wind_speed_unit": "ms"}),
    ):
        path = RAW / f"{tag}_{city}.csv"
        if path.exists() and not FORCE:
            print("[cache]", path)
            continue
        print("[unduh]", tag, city)
        fetch_city(url, vars_, lat, lon, tz, extra).to_csv(path, index=False)
print("Selesai.")

[cache] data/raw/aq_Jakarta.csv
[cache] data/raw/wx_Jakarta.csv
[cache] data/raw/aq_Bandung.csv
[cache] data/raw/wx_Bandung.csv
[cache] data/raw/aq_Semarang.csv
[cache] data/raw/wx_Semarang.csv
[cache] data/raw/aq_Surabaya.csv
[cache] data/raw/wx_Surabaya.csv
[cache] data/raw/aq_Medan.csv
[cache] data/raw/wx_Medan.csv
[cache] data/raw/aq_Makassar.csv
[cache] data/raw/wx_Makassar.csv
Selesai.


**Baca Data Mentah dan Agregasi Harian**

In [11]:
def load_hourly(tag):
    parts = []
    for city in CITIES:
        df = pd.read_csv(RAW / f"{tag}_{city}.csv", parse_dates=["time"])
        df.insert(0, "kota", city)
        parts.append(df)
    return pd.concat(parts, ignore_index=True)


def daily_air(aq):
    aq = aq.copy()
    aq["tanggal"] = aq["time"].dt.normalize()
    g = aq.groupby(["kota", "tanggal"])
    out = g[AQ_VARS].mean()
    out["pm2_5_maks"] = g["pm2_5"].max()
    return out.reset_index()


def daily_weather(wx):
    wx = wx.copy()
    wx["tanggal"] = wx["time"].dt.normalize()
    rad = np.deg2rad(wx["wind_direction_10m"])
    wx["u"] = -wx["wind_speed_10m"] * np.sin(rad)
    wx["v"] = -wx["wind_speed_10m"] * np.cos(rad)
    g = wx.groupby(["kota", "tanggal"])
    out = pd.DataFrame({
        "suhu_rata": g["temperature_2m"].mean(),
        "kelembapan_rata": g["relative_humidity_2m"].mean(),
        "curah_hujan_total": g["precipitation"].sum(min_count=1),
        "angin_rata": g["wind_speed_10m"].mean(),
        "tekanan_rata": g["surface_pressure"].mean(),
        "awan_rata": g["cloud_cover"].mean(),
        "u": g["u"].mean(), "v": g["v"].mean(),
    })
    out["arah_angin_dominan"] = np.degrees(np.arctan2(-out["u"], -out["v"])) % 360
    return out.drop(columns=["u", "v"]).reset_index()

**Data BPS (manual)**

In [12]:
BPS_MANUAL = {
    ("DKI Jakarta", 2023):      (2272301, 34877, 500146, 8889450),
    ("DKI Jakarta", 2024):      (2333391, 36381, 520051, 9167512),
    ("DKI Jakarta", 2025):      (2336960, 37981, 527841, 9376276),
    ("Jawa Barat", 2023):       (2819163, 33038, 747328, 22355333),
    ("Jawa Barat", 2024):       (2965471, 34520, 756257, 23348676),
    ("Jawa Barat", 2025):       (2980773, 36679, 779518, 24260590),
    ("Jawa Tengah", 2023):      (1627049, 36626, 673839, 18499683),
    ("Jawa Tengah", 2024):      (1683498, 38024, 685334, 19154994),
    ("Jawa Tengah", 2025):      (1741611, 39734, 700639, 19838639),
    ("Jawa Timur", 2023):       (5439502, 43261, 781907, 18885567),
    ("Jawa Timur", 2024):       (5544750, 45193, 797226, 19571995),
    ("Jawa Timur", 2025):       (5588804, 48012, 816582, 20246077),
    ("Sumatera Utara", 2023):   (812499, 9321, 317860, 6616681),
    ("Sumatera Utara", 2024):   (846797, 9784, 326686, 6953827),
    ("Sumatera Utara", 2025):   (872015, 10541, 333530, 7169777),
    ("Sulawesi Selatan", 2023): (604468, 5447, 230922, 4292112),
    ("Sulawesi Selatan", 2024): (631062, 5781, 239266, 4507930),
    ("Sulawesi Selatan", 2025): (660750, 6123, 249085, 4719350),
}

bps_path = RAW / "bps_kendaraan.csv"
rows = []
for p in sorted({c[3] for c in CITIES.values()}):
    for y in YEARS:
        v = BPS_MANUAL.get((p, y), (None,) * 4)
        rows.append((p, y, *[np.nan if a is None else a for a in v]))
bps = pd.DataFrame(rows, columns=BPS_COLS)
bps.to_csv(bps_path, index=False)
print(bps.to_string(index=False))
print("\nBaris yang masih kosong:", int(bps["bus"].isna().sum()), "dari", len(bps))

        provinsi  tahun  mobil_penumpang   bus   truk  sepeda_motor
     DKI Jakarta   2023          2272301 34877 500146       8889450
     DKI Jakarta   2024          2333391 36381 520051       9167512
     DKI Jakarta   2025          2336960 37981 527841       9376276
      Jawa Barat   2023          2819163 33038 747328      22355333
      Jawa Barat   2024          2965471 34520 756257      23348676
      Jawa Barat   2025          2980773 36679 779518      24260590
     Jawa Tengah   2023          1627049 36626 673839      18499683
     Jawa Tengah   2024          1683498 38024 685334      19154994
     Jawa Tengah   2025          1741611 39734 700639      19838639
      Jawa Timur   2023          5439502 43261 781907      18885567
      Jawa Timur   2024          5544750 45193 797226      19571995
      Jawa Timur   2025          5588804 48012 816582      20246077
Sulawesi Selatan   2023           604468  5447 230922       4292112
Sulawesi Selatan   2024           631062  5781 2

**Gabung Jadi Dataset Harian dan Ringkasan**

In [13]:
bps = pd.read_csv(bps_path)
jenis_cols = ["mobil_penumpang", "bus", "truk", "sepeda_motor"]
bps["total_kendaraan"] = bps[jenis_cols].sum(axis=1, min_count=4)
bps["proporsi_motor"] = bps["sepeda_motor"] / bps["total_kendaraan"]

aq_h, wx_h = load_hourly("aq"), load_hourly("wx")
df = daily_air(aq_h).merge(daily_weather(wx_h), on=["kota", "tanggal"], how="outer")
df["tahun"] = df["tanggal"].dt.year
df["bulan"] = df["tanggal"].dt.month
df["provinsi"] = df["kota"].map({k: v[3] for k, v in CITIES.items()})
df = df.merge(bps, on=["provinsi", "tahun"], how="left")
df = df.sort_values(["kota", "tanggal"]).reset_index(drop=True)
df.to_csv(PROC / "dataset_harian.csv", index=False)

print("Periode       :", df["tanggal"].min().date(), "s.d.", df["tanggal"].max().date())
print("Kota          :", df["kota"].nunique())
print("Obs per jam   : AQ =", len(aq_h), "| cuaca =", len(wx_h))
print("Obs harian    :", len(df))
print("Jumlah kolom  :", df.shape[1])
print("Duplikat      :", df.duplicated(["kota", "tanggal"]).sum())
print("\nData hilang (%):")
print((df.isna().mean() * 100).round(2).to_string())

Periode       : 2023-01-01 s.d. 2025-12-31
Kota          : 6
Obs per jam   : AQ = 157824 | cuaca = 157824
Obs harian    : 6576
Jumlah kolom  : 27
Duplikat      : 0

Data hilang (%):
kota                     0.0
tanggal                  0.0
pm2_5                    0.0
pm10                     0.0
carbon_monoxide          0.0
nitrogen_dioxide         0.0
sulphur_dioxide          0.0
ozone                    0.0
aerosol_optical_depth    0.0
us_aqi                   0.0
pm2_5_maks               0.0
suhu_rata                0.0
kelembapan_rata          0.0
curah_hujan_total        0.0
angin_rata               0.0
tekanan_rata             0.0
awan_rata                0.0
arah_angin_dominan       0.0
tahun                    0.0
bulan                    0.0
provinsi                 0.0
mobil_penumpang          0.0
bus                      0.0
truk                     0.0
sepeda_motor             0.0
total_kendaraan          0.0
proporsi_motor           0.0
